# VulnGNN Phase 2B: Kaggle graph extraction

This notebook defaults to a one-batch smoke run (`BATCH_LIMIT = 1`). The complete manifest contains 117,673 functions in 59 batches of 2,000 (last batch 1,673). Never use `/kaggle/working` as durable storage: Kaggle currently has **No persistence** selected for this workflow, and stopping the session may discard the workset.

For the one-batch compatibility smoke, reuse Kaggle’s installed pandas/PyArrow; for larger prefixes, install and verify the packaged pinned environment, attach the private Linux x86_64 inputs Dataset, and (when resuming) attach the latest private checkpoint Dataset. Set `BATCH_LIMIT` to a cumulative prefix (10, then 20, and so on through 59); each run validates and skips completed batches. After every chunk, verify the `.tar.gz.sha256` sidecar and save both files as a new private Kaggle Dataset version or download them before stopping. On the next session, restore the latest archive with `scripts/restore_graph_checkpoint.py` before running the next larger prefix. Do not start full extraction until the manual steps in README are complete.

## Verified Kaggle smoke evidence (batch 00000 only)

The latest verified Kaggle CPU smoke (notebook version 4, 2026-11-16) ran only batch `00000` (2,000 functions) on Linux x86_64 with 4 CPUs, Python 3.13.18, pandas 2.3.3, PyArrow 23.0.1, OpenJDK 21.0.12, and Joern 4.0.665. It emitted 1,830 graphs (1,797 `ok`, 33 `ok_truncated`), with 170 `no_method` and zero `batch_error`: safe coverage 1,686/1,842 (91.3%) and vulnerable coverage 144/158 (91.1%). Parse/export/conversion/batch times were 25.829/7.216/12.481/46.629 seconds; sampled peak process-tree RSS was 1,217.13 MiB (250 ms sampling). The seeded 20-graph integrity check passed. Resume validated and skipped the batch, preserving graph JSONL SHA-256 `53524c780bbb89df3db8eeb316155b61a3692bce5f2d26f344d27cc7cd7f974d`. The checkpoint archive SHA-256 was `eb0dfe380f7340b46da8f17cf8f4f75c23b4f20b242ab7518a1c97cfc3d04039`; after downloading its archive and sidecar, a clean-workset restore passed and the local archive was reverified after stopping the Kaggle session. This verifies one-batch compatibility and checkpoint recovery only, not a full extraction or runtime guarantee.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import tarfile
import time

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

INPUT_ROOT = Path("/kaggle/input")
PACKAGE_NAME = "vulngnn-phase2b-kaggle-v4.tar.gz.bin"
EXPECTED_PACKAGE_SHA256 = "c13c5587d2bd6249aa3481cdb4de868d50588bdc2ba8749dbecdc8ed22ff314f"
BASE_INPUT = Path("/kaggle/working/vulngnn-phase2b-inputs")
WORKSET = Path("/kaggle/working/graph_workset")
BATCH_LIMIT = 1  # Smoke default; cumulative prefixes: 10, 20, 30, 40, 50, then 59.
CHECKPOINT_ARCHIVE = Path(f"/kaggle/working/graphs-through-{BATCH_LIMIT:02d}-{int(time.time())}.tar.gz")
REQUIRED_FILES = (
    "data/processed/bigvul_clean.parquet",
    "data/graph_workset/manifest.csv",
    "data/graph_workset/coverage_by_project.csv",
    "data/graph_workset/workset_metadata.json",
    "tools/joern/joern-cli/joern-parse",
    "tools/joern/joern-cli/joern-export",
    "scripts/run_graph_pipeline.py",
    "scripts/restore_graph_checkpoint.py",
    "src/vulngnn/graphs/run_cpg_batches.py",
    "requirements-graph.txt",
)

if platform.system() != "Linux" or platform.machine().lower() not in {"x86_64", "amd64"}:
    raise RuntimeError(f"Expected Linux x86_64; got {platform.system()} {platform.machine()}")
print("Platform:", platform.platform())

# Keep the uploaded tarball under a .bin suffix so Kaggle mounts the raw bytes.
# Verify its streamed SHA-256 before extracting any package contents.
package_matches = list(INPUT_ROOT.rglob(PACKAGE_NAME))
if len(package_matches) != 1:
    raise RuntimeError(f"Expected exactly one mounted raw package file named {PACKAGE_NAME}; found {package_matches}")
PACKAGE_ARCHIVE = package_matches[0]
actual_package_sha256 = sha256_file(PACKAGE_ARCHIVE)
print("Mounted package archive:", PACKAGE_ARCHIVE, f"({PACKAGE_ARCHIVE.stat().st_size:,} bytes)")
print("Mounted package SHA-256:", actual_package_sha256)
if actual_package_sha256 != EXPECTED_PACKAGE_SHA256:
    raise RuntimeError(f"Package SHA-256 mismatch: expected {EXPECTED_PACKAGE_SHA256}, got {actual_package_sha256}")

mac_magic = {bytes.fromhex(x) for x in ("feedface", "feedfacf", "cefaedfe", "cffaedfe", "cafebabe", "bebafeca")}
def audit_package_tree(root, check_permissions=False):
    missing = [name for name in REQUIRED_FILES if not (root / name).is_file()]
    if missing:
        raise RuntimeError(f"Package missing required files under {root}: {missing}")
    files = [path for path in root.rglob("*") if path.is_file()]
    cache_files = [str(path.relative_to(root)) for path in files
                   if "__pycache__" in path.relative_to(root).parts or path.suffix == ".pyc"]
    if cache_files:
        raise RuntimeError(f"Package unexpectedly contains Python cache files: {cache_files[:10]}")
    csv_files = [(path, path.stat().st_size) for path in files if path.suffix.lower() == ".csv"]
    oversized_csv = [(str(path.relative_to(root)), size) for path, size in csv_files if size > 100 * 1024**2]
    if oversized_csv:
        raise RuntimeError(f"Unexpected large CSV inputs: {oversized_csv}")
    print("CSV package members:", [(str(path.relative_to(root)), size) for path, size in csv_files])
    joern_root = root / "tools/joern/joern-cli"
    joern_files = [path for path in joern_root.rglob("*") if path.is_file()]
    mach_o = []
    elf = []
    for path in joern_files:
        with path.open("rb") as stream:
            signature = stream.read(4)
        if signature in mac_magic:
            mach_o.append(str(path.relative_to(root)))
        if signature == b"\x7fELF":
            elf.append(path)
    mac_named = [str(path.relative_to(root)) for path in joern_files
                 if "macos-arm" in path.name.lower() or "-macos" in path.name.lower()]
    if mach_o or mac_named or not elf:
        raise RuntimeError(f"Expected Linux Joern executables; Mach-O={mach_o}, macOS names={mac_named}, ELF count={len(elf)}")
    for name in ("joern-parse", "joern-export"):
        launcher = joern_root / name
        if not launcher.read_text(errors="replace").startswith("#!/usr/bin/env sh"):
            raise RuntimeError(f"Invalid Linux Joern launcher: {launcher}")
        if check_permissions and not os.access(launcher, os.X_OK):
            raise RuntimeError(f"Joern launcher is not executable after staging: {launcher}")
    if check_permissions and any(not os.access(path, os.X_OK) for path in elf):
        raise RuntimeError("One or more Joern ELF helpers are not executable after staging")
    print("Joern platform audit: ELF count", len(elf), "Mach-O count", len(mach_o))
    print(f"Package tree audit: PASS ({len(files):,} files); no cache files or oversized CSVs")

def restore_joern_permissions(root):
    joern_root = root / "tools/joern/joern-cli"
    for path in (p for p in joern_root.rglob("*") if p.is_file()):
        try:
            with path.open("rb") as stream:
                signature = stream.read(4)
        except OSError:
            continue
        if signature == b"\x7fELF" or path.suffix == ".sh" or path.name in {"joern", "joern-parse", "joern-export"}:
            path.chmod(path.stat().st_mode | 0o111)

if package_matches:
    PACKAGE_ARCHIVE = package_matches[0]
    print("Attached Kaggle archive:", PACKAGE_ARCHIVE, f"({PACKAGE_ARCHIVE.stat().st_size:,} bytes)")
    with tarfile.open(PACKAGE_ARCHIVE, "r:gz") as archive:
        members = archive.getmembers()
        member_names = {member.name for member in members}
        missing = sorted(set(REQUIRED_FILES) - member_names)
        if missing:
            raise RuntimeError(f"Archive missing required members: {missing}")
        if any(Path(name).is_absolute() or ".." in Path(name).parts for name in member_names):
            raise RuntimeError("Unsafe archive member path")
        if any("__pycache__" in Path(name).parts or name.endswith(".pyc") for name in member_names):
            raise RuntimeError("Package unexpectedly contains Python cache files")
        csv_members = [(member.name, member.size) for member in members if member.name.lower().endswith(".csv")]
        if any(size > 100 * 1024**2 for _, size in csv_members):
            raise RuntimeError(f"Unexpected large CSV input members: {csv_members}")
        if any("macos-arm" in name or "-macos" in name for name in member_names
               if name.startswith("tools/joern/")):
            raise RuntimeError("Linux archive contains macOS-specific Joern filenames")
        native_executables = [member for member in members
                              if member.isfile() and member.name.startswith("tools/joern/joern-cli/")
                              and member.mode & 0o111]
        archive_mach_o = []
        archive_elf = []
        for member in native_executables:
            signature = archive.extractfile(member).read(4)
            if signature in mac_magic:
                archive_mach_o.append(member.name)
            if signature == b"\x7fELF":
                archive_elf.append(member.name)
        if archive_mach_o or not archive_elf:
            raise RuntimeError(f"Archive Joern binary audit failed: Mach-O={archive_mach_o}, ELF={len(archive_elf)}")
        print("Archive members:", len(members), "CSV:", csv_members,
              "Joern ELF:", len(archive_elf), "Mach-O:", len(archive_mach_o))
        metadata = json.load(archive.extractfile("data/graph_workset/workset_metadata.json"))
        if metadata["functions"] != 117_673 or metadata["batch_size"] != 2_000:
            raise RuntimeError(f"Unexpected workset metadata: {metadata}")
        print(f"Archive integrity: PASS ({len(members):,} members); workset metadata:", metadata)
        if not (BASE_INPUT / REQUIRED_FILES[0]).is_file():
            BASE_INPUT.mkdir(parents=True, exist_ok=True)
            archive.extractall(BASE_INPUT)
else:
    unpacked_roots = []
    for requirements_file in INPUT_ROOT.rglob("requirements-graph.txt"):
        root = requirements_file.parent
        if all((root / name).is_file() for name in REQUIRED_FILES):
            unpacked_roots.append(root)
    unpacked_roots = sorted(set(unpacked_roots))
    if len(unpacked_roots) != 1:
        raise RuntimeError(f"Expected one Kaggle-extracted package tree; found {unpacked_roots}")
    PACKAGE_ROOT = unpacked_roots[0]
    print("Kaggle unpacked the uploaded archive; mounted package root:", PACKAGE_ROOT)
    audit_package_tree(PACKAGE_ROOT)
    metadata = json.loads((PACKAGE_ROOT / "data/graph_workset/workset_metadata.json").read_text())
    if metadata["functions"] != 117_673 or metadata["batch_size"] != 2_000:
        raise RuntimeError(f"Unexpected workset metadata: {metadata}")
    if not (BASE_INPUT / REQUIRED_FILES[0]).is_file():
        shutil.copytree(PACKAGE_ROOT, BASE_INPUT)
        print("Copied the read-only Kaggle input to writable working storage:", BASE_INPUT)

restore_joern_permissions(BASE_INPUT)
audit_package_tree(BASE_INPUT, check_permissions=True)

# Reuse Kaggle's installed data libraries for all prefixes; record exact compatibility versions.
def install_graph_requirements():
    requirements_log = Path("/kaggle/working/python_requirements_install.log")
    with requirements_log.open("w") as log:
        result = subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                                 str(BASE_INPUT / "requirements-graph.txt")],
                                stdout=log, stderr=subprocess.STDOUT, text=True)
    if result.returncode:
        print(requirements_log.read_text(errors="replace"))
        raise subprocess.CalledProcessError(result.returncode, result.args)
    print("Installed packaged Python dependencies; full log:", requirements_log)

if BATCH_LIMIT == 1:
    try:
        import pandas as pd
        import pyarrow
        import pyarrow.parquet as pq
    except ImportError:
        install_graph_requirements()
        import pandas as pd
        import pyarrow
        import pyarrow.parquet as pq
    print("Compatibility smoke runtime:", pd.__version__, pyarrow.__version__)
else:
    # Kaggle does not provide network access in this session. Reuse the same
    # pandas/PyArrow versions validated by the compatibility smoke.
    import pandas as pd
    import pyarrow
    import pyarrow.parquet as pq
    runtime_versions = {"pandas": pd.__version__, "pyarrow": pyarrow.__version__}
    print("Kaggle installed compatibility versions:", runtime_versions)
DATA = BASE_INPUT / "data/processed/bigvul_clean.parquet"
JOERN = BASE_INPUT / "tools/joern/joern-cli"
BASE_MANIFEST = BASE_INPUT / "data/graph_workset/manifest.csv"
BASE_COVERAGE = BASE_INPUT / "data/graph_workset/coverage_by_project.csv"
BASE_METADATA = BASE_INPUT / "data/graph_workset/workset_metadata.json"
PROJECT = BASE_INPUT
for expected in (DATA, BASE_MANIFEST, BASE_COVERAGE, BASE_METADATA,
                 PROJECT / "scripts/run_graph_pipeline.py"):
    if not expected.is_file():
        raise FileNotFoundError(expected)

# Locate a Java 21 runtime or provision the official OpenJDK 21 package.
def java_major(java_path):
    result = subprocess.run([str(java_path), "-version"], capture_output=True, text=True)
    text = result.stdout + result.stderr
    match = re.search(r'(?:version\s+[" ]?|openjdk\s+)([0-9]+)', text, re.I)
    return (int(match.group(1)), text) if match else (None, text)

java_candidates = []
if shutil.which("java"):
    java_candidates.append(Path(shutil.which("java")))
java_candidates.extend(Path(p) for p in (
    "/usr/lib/jvm/java-21-openjdk-amd64/bin/java",
    "/usr/lib/jvm/java-21-openjdk/bin/java",
))
java_candidates.extend(Path(p) for p in Path("/usr/lib/jvm").glob("*/bin/java") if Path("/usr/lib/jvm").exists())
JAVA_BIN = None
JAVA_TEXT = ""
for candidate in dict.fromkeys(java_candidates):
    if candidate.is_file():
        major, text = java_major(candidate)
        if major == 21:
            JAVA_BIN, JAVA_TEXT = candidate.resolve(), text
            break
if JAVA_BIN is None:
    if not shutil.which("apt-get"):
        raise RuntimeError("Java 21 not found and apt-get is unavailable")
    java_install_log = Path("/kaggle/working/java21_install.log")
    with java_install_log.open("w") as log:
        for install_command in (["apt-get", "update"],
                                ["apt-get", "install", "-y", "openjdk-21-jdk-headless"]):
            result = subprocess.run(install_command, stdout=log, stderr=subprocess.STDOUT, text=True)
            if result.returncode:
                print(java_install_log.read_text(errors="replace"))
                raise subprocess.CalledProcessError(result.returncode, install_command)
    for candidate in Path("/usr/lib/jvm").glob("*/bin/java"):
        major, text = java_major(candidate)
        if major == 21:
            JAVA_BIN, JAVA_TEXT = candidate.resolve(), text
            break
if JAVA_BIN is None:
    raise RuntimeError("OpenJDK 21 installation/lookup failed")
JAVA_HOME = JAVA_BIN.parent.parent
os.environ["JAVA_HOME"] = str(JAVA_HOME)
os.environ["PATH"] = str(JAVA_HOME / "bin") + os.pathsep + os.environ.get("PATH", "")
print("JAVA_HOME:", JAVA_HOME)
print("Java executable:", JAVA_BIN)
print("$ java -version")
print(JAVA_TEXT.strip())

# Confirm the bundled Joern release and Linux shell launchers.
version_jars = sorted((JOERN / "lib").glob("io.joern.joern-cli-*.jar"))
if len(version_jars) != 1 or "4.0.665" not in version_jars[0].name:
    raise RuntimeError(f"Expected Joern v4.0.665; found {[p.name for p in version_jars]}")
for name in ("joern-parse", "joern-export"):
    launcher = JOERN / name
    if not launcher.is_file() or not os.access(launcher, os.X_OK):
        raise RuntimeError(f"Joern launcher is missing or not executable: {launcher}")
    if not launcher.read_text(errors="replace").startswith("#!/usr/bin/env sh"):
        raise RuntimeError(f"Unexpected Linux launcher format: {launcher}")
    result = subprocess.run([str(launcher), "--help"], env=os.environ.copy(),
                            capture_output=True, text=True)
    help_log = Path(f"/kaggle/working/{name.replace('-', '_')}_help.log")
    help_log.write_text(result.stdout + result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"{name} --help failed ({result.returncode}); full log: {help_log}\n{result.stdout}\n{result.stderr}")
    print(f"{name} --help: PASS; full output: {help_log}")
print("Joern version:", version_jars[0].name)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__, "pyarrow:", pyarrow.__version__)
parquet_schema = pq.ParquetFile(DATA).schema.names
required_columns = {"vul", "project_g", "func_before", "norm_hash", "near_dup_cross", "near_dup_within"}
if not required_columns.issubset(parquet_schema):
    raise RuntimeError(f"Packaged parquet schema missing {required_columns - set(parquet_schema)}")
parquet_check = pd.read_parquet(DATA, columns=["vul", "project_g", "near_dup_cross", "near_dup_within"])
assert len(parquet_check) == 182_100 and int(parquet_check.vul.sum()) == 8_609
assert parquet_check.project_g.nunique() == 295
assert str(parquet_check.near_dup_cross.dtype) == "bool"
assert str(parquet_check.near_dup_within.dtype) == "bool"
parquet_sha256 = sha256_file(DATA)
assert parquet_sha256 == metadata["source_parquet_sha256"]
print("Packaged parquet schema/counts/hash: PASS", parquet_sha256)
del parquet_check

# Actual Kaggle resource snapshot before the batch.
def memory_snapshot():
    values = {}
    for line in Path("/proc/meminfo").read_text().splitlines():
        key, rest = line.split(":", 1)
        if key in {"MemTotal", "MemAvailable"}:
            values[key] = int(rest.strip().split()[0]) * 1024
    return values

resources_before = {
    "cpu_count": os.cpu_count(),
    "memory_bytes": memory_snapshot(),
    "working_disk_total_bytes": shutil.disk_usage("/kaggle/working").total,
    "working_disk_free_bytes": shutil.disk_usage("/kaggle/working").free,
    "accelerator": "CPU requested; this notebook does not use CUDA",
}
print("Kaggle resources before extraction:", json.dumps(resources_before, indent=2))
print(f"Execution is limited to batches with batch_id < {BATCH_LIMIT}; the default remains the batch-00000 smoke.")


In [ ]:
# Seed or reuse the exact prepared manifest; the default prefix contains batch 00000 only.
if not (WORKSET / "manifest.csv").is_file():
    WORKSET.mkdir(parents=True, exist_ok=True)
    shutil.copy2(BASE_MANIFEST, WORKSET / "manifest.csv")
    shutil.copy2(BASE_COVERAGE, WORKSET / "coverage_by_project.csv")
    shutil.copy2(BASE_METADATA, WORKSET / "workset_metadata.json")

manifest = pd.read_csv(WORKSET / "manifest.csv", dtype={"uid": str, "norm_hash": str},
                       usecols=["uid", "norm_hash", "ext", "batch_id", "vul", "project_g"])
assert len(manifest) == 117_673 and manifest.uid.nunique() == 117_673
assert int(manifest.batch_id.max()) == 58 and manifest.batch_id.nunique() == 59
batch_manifest = manifest.loc[manifest.batch_id == 0].copy()
assert len(batch_manifest) == 2_000 and batch_manifest.uid.nunique() == 2_000
assert set(batch_manifest.ext) <= {".c", ".cpp"}
assert all(uid == "f_" + norm_hash for uid, norm_hash in
           zip(batch_manifest.uid, batch_manifest.norm_hash))
BATCH_MANIFEST_PATH = WORKSET / "batch_00000_manifest.csv"
batch_manifest.to_csv(BATCH_MANIFEST_PATH, index=False)
selected_manifest = manifest.loc[manifest.batch_id < BATCH_LIMIT].copy()
assert 1 <= BATCH_LIMIT <= 59 and selected_manifest.batch_id.nunique() == BATCH_LIMIT
expected_run_uids = set(selected_manifest.uid)
expected_uids = set(batch_manifest.uid)
expected_uid_sha256 = hashlib.sha256("\n".join(sorted(expected_uids)).encode()).hexdigest()
print("Manifest verified:", len(manifest), "rows; 59 batches")
print("Selected prefix:", BATCH_LIMIT, "batches;", len(selected_manifest), "functions")
print("Batch 00000:", len(batch_manifest), "unique UIDs; UID SHA-256:", expected_uid_sha256)
print("Class counts:", batch_manifest.vul.value_counts().sort_index().to_dict())
print("Extensions:", batch_manifest.ext.value_counts().to_dict())

# Optional checkpoint input: verify archive checksum, internal member hashes,
# and exact workset identity before restoring any files.
checkpoint_inputs = sorted(
    set(INPUT_ROOT.rglob("graphs-through-*.tar.gz"))
    | set(INPUT_ROOT.rglob("graphs-through-*.tar.gz.bin"))
)
if len(checkpoint_inputs) > 1:
    raise RuntimeError(f"Attach exactly one latest checkpoint Dataset; found {checkpoint_inputs}")
if checkpoint_inputs:
    restore_command = [sys.executable, str(PROJECT / "scripts/restore_graph_checkpoint.py"),
                       str(checkpoint_inputs[0]), "--workset", str(WORKSET)]
    print("$", " ".join(restore_command))
    subprocess.run(restore_command, cwd=PROJECT, check=True)

In [ ]:
# Execute the configured cumulative prefix and sample the runner plus descendants for peak RSS.
env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT / "src") + os.pathsep + env.get("PYTHONPATH", "")
env["JAVA_HOME"] = str(JAVA_HOME)
env["PATH"] = str(JAVA_HOME / "bin") + os.pathsep + env.get("PATH", "")
command = [
    sys.executable, str(PROJECT / "scripts/run_graph_pipeline.py"),
    "--data", str(DATA), "--workset", str(WORKSET),
    "--joern-dir", str(JOERN), "--java-home", str(JAVA_HOME),
    "--negative-ratio", "5.0", "--batch-size", "2000", "--max-nodes", "500",
    "--limit-batches", str(BATCH_LIMIT),
]
run_log = Path(f"/kaggle/working/pipeline_through_{BATCH_LIMIT:02d}.log")
print("$", " ".join(command), flush=True)
started = time.perf_counter()
peak_rss_kib = 0
rss_samples = 0
sampling_error = None
with run_log.open("w") as log:
    process = subprocess.Popen(command, cwd=PROJECT, env=env, stdout=log,
                               stderr=subprocess.STDOUT)
    while process.poll() is None:
        try:
            raw = subprocess.check_output(["ps", "-eo", "pid=,ppid=,rss="], text=True)
            rows = {}
            for line in raw.splitlines():
                fields = line.split()
                if len(fields) == 3:
                    try:
                        rows[int(fields[0])] = (int(fields[1]), int(fields[2]))
                    except ValueError:
                        pass
            descendants = {process.pid}
            changed = True
            while changed:
                changed = False
                for pid, (parent, rss) in rows.items():
                    if parent in descendants and pid not in descendants:
                        descendants.add(pid)
                        changed = True
            peak_rss_kib = max(peak_rss_kib,
                               sum(rows[pid][1] for pid in descendants if pid in rows))
            rss_samples += 1
        except Exception as exc:
            sampling_error = f"{type(exc).__name__}: {exc}"
        time.sleep(0.25)
    return_code = process.wait()
wall_seconds = time.perf_counter() - started
log_text = run_log.read_text(errors="replace")
print("Batch pipeline log tail:")
print("\n".join(log_text.splitlines()[-50:]))
print("Pipeline return code:", return_code, "wall seconds:", round(wall_seconds, 3))
if sampling_error:
    print("RSS sampling issue:", sampling_error)
if return_code != 0:
    raise RuntimeError(f"Batch 00000 failed with exit code {return_code}; full log: {run_log}")

summary = json.loads((WORKSET / "graphs/audit_summary.json").read_text())
stages = pd.read_csv(WORKSET / "graphs/stage_timings.csv")
stats = pd.read_csv(WORKSET / "graphs/coverage.csv", dtype={"uid": str})
coverage = stats.copy()
assert set(stats.uid) == expected_run_uids and stats.uid.nunique() == len(selected_manifest)
assert summary["planned_functions"] == 117_673
assert summary["audited_functions"] == len(selected_manifest)
assert summary["error_count"] == 0
assert int((stats.status == "batch_error").sum()) == 0

disk_after = shutil.disk_usage("/kaggle/working")
prefix_stages = stages.loc[stages.batch_id < BATCH_LIMIT]
stage_totals = {name: pd.to_numeric(prefix_stages[name], errors="coerce").sum(min_count=1)
                for name in ("parse_seconds", "export_seconds", "conversion_seconds",
                             "batch_seconds", "csv_export_bytes")}
status_counts = stats.status.value_counts().to_dict()
compact_paths = [WORKSET / "graphs" / f"batch_{batch_id:05d}.jsonl"
                 for batch_id in range(BATCH_LIMIT)
                 if (WORKSET / "graphs" / f"batch_{batch_id:05d}.jsonl").is_file()]
compact_bytes = sum(path.stat().st_size for path in compact_paths)
metrics = {
    "platform": platform.platform(),
    "cpu_count": resources_before["cpu_count"],
    "memory_before_bytes": resources_before["memory_bytes"],
    "working_disk_total_bytes": resources_before["working_disk_total_bytes"],
    "working_disk_free_before_bytes": resources_before["working_disk_free_bytes"],
    "working_disk_free_after_bytes": disk_after.free,
    "parquet_sha256": parquet_sha256,
    "python": sys.version.split()[0],
    "java_home": str(JAVA_HOME),
    "java_binary": str(JAVA_BIN),
    "java_version": JAVA_TEXT.strip(),
    "joern_version": version_jars[0].name,
    "selected_batch_limit": BATCH_LIMIT,
    "functions": len(stats),
    "batch_00000_successful_graphs": int(pd.read_csv(WORKSET / "graphs/batch_00000_stats.csv").status.isin(["ok", "ok_truncated"]).sum()),
    "uid_sha256": expected_uid_sha256,
    "status_counts": status_counts,
    "successful_graphs": int(stats.status.isin(["ok", "ok_truncated"]).sum()),
    "failed_graphs": int((~stats.status.isin(["ok", "ok_truncated"])).sum()),
    "truncated_graphs": int(stats.truncated.sum()),
    "parse_seconds": float(stage_totals["parse_seconds"]),
    "export_seconds": float(stage_totals["export_seconds"]),
    "conversion_seconds": float(stage_totals["conversion_seconds"]),
    "batch_seconds": float(stage_totals["batch_seconds"]),
    "pipeline_wall_seconds_including_audit": wall_seconds,
    "csv_export_bytes_recorded": int(stage_totals["csv_export_bytes"]),
    "compact_graph_bytes": compact_bytes,
    "peak_process_tree_rss_mib": peak_rss_kib / 1024,
    "rss_sampling_interval_seconds": 0.25,
    "rss_sample_count": rss_samples,
    "rss_sampling_error": sampling_error,
    "audit": summary,
}
METRICS_PATH = WORKSET / "graphs" / f"kaggle_metrics_through_{BATCH_LIMIT:02d}.json"
METRICS_PATH.write_text(json.dumps(metrics, indent=2) + "\n")
print("Actual Kaggle batch metrics:")
print(json.dumps(metrics, indent=2))

# Coverage is reported for every class/extension, including failed functions.
stats_for_coverage = stats[["uid", "status"]].merge(
    selected_manifest[["uid", "vul", "ext"]], on="uid", validate="one_to_one")
coverage_table = (stats_for_coverage.assign(label=stats_for_coverage.vul.map({0: "Safe", 1: "Vulnerable"}),
                                             extension=stats_for_coverage.ext.map({".c": "C", ".cpp": "C++"}))
                  .assign(success=lambda x: x.status.isin(["ok", "ok_truncated"]))
                  .groupby(["label", "extension"], dropna=False)
                  .agg(functions=("uid", "size"), graphs=("success", "sum"))
                  .reset_index())
coverage_table["coverage"] = coverage_table.graphs / coverage_table.functions
print("Coverage by class and extension:")
print(coverage_table.to_string(index=False))
coverage_table.to_csv(WORKSET / "graphs" / "kaggle_coverage_by_class_extension.csv", index=False)
print("Per-function statuses/failures:", WORKSET / "graphs/coverage.csv")

In [ ]:
# Validate 20 seeded successful graphs, checkpoint the batch, then prove resume skips it.
import random

stats_index = stats.set_index("uid")
manifest_index = selected_manifest.set_index("uid")
compact_paths = [WORKSET / "graphs" / f"batch_{batch_id:05d}.jsonl"
                 for batch_id in range(BATCH_LIMIT)]
successful_uids = stats.loc[stats.status.isin(["ok", "ok_truncated"]), "uid"].tolist()
quality_uids = set(random.Random(42).sample(successful_uids, min(20, len(successful_uids))))
graph_uids = set()
quality_graphs = {}
for compact_file in compact_paths:
    with compact_file.open() as stream:
        for line in stream:
            graph = json.loads(line)
            uid = graph["uid"]
            if uid in graph_uids:
                raise AssertionError(f"Duplicate graph UID: {uid}")
            graph_uids.add(uid)
            if uid in quality_uids:
                quality_graphs[uid] = graph
assert graph_uids == set(successful_uids)
assert set(quality_graphs) == quality_uids
quality_rows = []
allowed_edge_types = ["AST", "CFG", "REACHING_DEF", "CDG"]
for uid in sorted(quality_uids):
    graph = quality_graphs[uid]
    planned = manifest_index.loc[uid]
    record = stats_index.loc[uid]
    source_name = f"{uid}{planned.ext}"
    source_path = WORKSET / "batches" / f"batch_{int(planned.batch_id):05d}" / "src" / source_name
    if source_path.is_file():
        source = source_path.read_text(errors="replace")
    else:
        # A checkpoint intentionally omits all source snippets. Re-read only
        # sampled source rows when validating graphs in a fresh session.
        source_table = pq.read_table(DATA, columns=["norm_hash", "func_before"],
                                     filters=[[('norm_hash', '==', str(planned.norm_hash))]])
        if source_table.num_rows != 1:
            raise RuntimeError(f"Expected one parquet source for {uid}; found {source_table.num_rows}")
        source = source_table.column("func_before")[0].as_py()
    source_line_count = max(1, len(source.splitlines()))
    assert uid == "f_" + str(planned.norm_hash)
    assert graph["norm_hash"] == planned.norm_hash
    assert int(graph["label"]) == int(planned.vul)
    assert graph["source_file"] == source_name
    assert len(graph["nodes"]) > 0 and len(graph["edges"]) > 0
    assert graph["edge_types"] == allowed_edge_types
    assert all(node["type"] in json.loads(record.node_types) for node in graph["nodes"])
    source_files = json.loads(record.source_filenames)
    assert source_files and all(Path(name).name == source_name for name in source_files), source_files
    assert all(edge[2] in range(len(allowed_edge_types)) for edge in graph["edges"])
    assert all(0 <= edge[0] < len(graph["nodes"]) and 0 <= edge[1] < len(graph["nodes"])
               for edge in graph["edges"])
    lines = [node["line"] for node in graph["nodes"] if node["line"] is not None]
    assert lines and all(isinstance(line, int) and 1 <= line <= source_line_count for line in lines)
    assert graph["truncated"] == (graph["original_nodes"] > graph["max_nodes"])
    assert graph["nodes_dropped"] == graph["original_nodes"] - len(graph["nodes"])
    assert bool(graph["truncated"]) == bool(record.truncated)
    quality_rows.append({"uid": uid, "norm_hash": graph["norm_hash"],
                         "label": graph["label"], "nodes": len(graph["nodes"]),
                         "edges": len(graph["edges"]), "truncated": graph["truncated"],
                         "source_file": source_name, "source_filenames": source_files})
quality = {"seed": 42, "inspected_graphs": len(quality_rows),
           "checks": ["uid/norm_hash/label/source match manifest", "positive nodes and edges",
                      "valid edge types and bounds", "line numbers preserved and within source",
                      "selected method subtree source filenames belong to the expected function",
                      "truncation metadata matches node counts"],
           "result": "PASS", "graphs": quality_rows}
quality_path = WORKSET / "graphs/quality_check.json"
quality_path.write_text(json.dumps(quality, indent=2) + "\n")
print("20-graph quality check:", quality_path, "PASS")

# The second call requests the same cumulative prefix and must validate/skip every completed batch.
graph_hashes_before = {batch_id: sha256_file(WORKSET / "graphs" / f"batch_{batch_id:05d}.jsonl")
                       for batch_id in range(BATCH_LIMIT)}
uid_set_before = graph_uids
resume_log = Path(f"/kaggle/working/resume_through_{BATCH_LIMIT:02d}.log")
resume_command = [sys.executable, "-m", "vulngnn.graphs.run_cpg_batches",
                  "--data", str(DATA), "--workset", str(WORKSET),
                  "--joern-dir", str(JOERN), "--java-home", str(JAVA_HOME),
                  "--max-nodes", "500", "--limit-batches", str(BATCH_LIMIT), "--resume"]
with resume_log.open("w") as log:
    resume_result = subprocess.run(resume_command, cwd=PROJECT, env=env,
                                   stdout=log, stderr=subprocess.STDOUT)
resume_text = resume_log.read_text(errors="replace")
print("Resume log tail:")
print("\n".join(resume_text.splitlines()[-30:]))
assert resume_result.returncode == 0
assert "resuming: validated prior outputs" in resume_text
graph_hashes_after = {batch_id: sha256_file(WORKSET / "graphs" / f"batch_{batch_id:05d}.jsonl")
                      for batch_id in range(BATCH_LIMIT)}
assert graph_hashes_after == graph_hashes_before
resumed_uids = {json.loads(line)["uid"] for path in compact_paths for line in path.open() if line.strip()}
assert resumed_uids == uid_set_before and len(resumed_uids) == int(metrics["successful_graphs"])
resume_evidence = {"resume_command": resume_command,
                   "completed_batches_skipped": True,
                   "resume_log_contains_validation": True,
                   "graph_sha256_by_batch_unchanged": graph_hashes_before,
                   "uid_count": len(resumed_uids)}
(WORKSET / "graphs/resume_check.json").write_text(
    json.dumps(resume_evidence, indent=2) + "\n")
print("Resume verification: PASS", json.dumps(resume_evidence, indent=2))

# Make the restart archive only after the run log, sample checks and resume check exist.
import shutil
pipeline_log_in_workset = WORKSET / "logs" / f"pipeline_through_{BATCH_LIMIT:02d}.log"
shutil.copy2(run_log, pipeline_log_in_workset)
shutil.copy2(resume_log, WORKSET / "logs" / f"resume_through_{BATCH_LIMIT:02d}.log")
sys.path.insert(0, str(PROJECT))
from scripts.run_graph_pipeline import make_archive
make_archive(WORKSET, CHECKPOINT_ARCHIVE)
checkpoint_sha = sha256_file(CHECKPOINT_ARCHIVE)
sidecar = CHECKPOINT_ARCHIVE.with_name(CHECKPOINT_ARCHIVE.name + ".sha256")
assert sidecar.is_file() and sidecar.read_text().split()[0] == checkpoint_sha
verify_command = [sys.executable, str(PROJECT / "scripts/restore_graph_checkpoint.py"),
                  str(CHECKPOINT_ARCHIVE), "--workset", str(WORKSET), "--verify-only"]
verify_result = subprocess.run(verify_command, cwd=PROJECT, env=env,
                               capture_output=True, text=True)
print("$", " ".join(verify_command))
print(verify_result.stdout, verify_result.stderr)
assert verify_result.returncode == 0
print("Durable checkpoint verified:", CHECKPOINT_ARCHIVE,
      f"({CHECKPOINT_ARCHIVE.stat().st_size:,} bytes)", "sha256:", checkpoint_sha)
print("Save/download the checkpoint archive AND its .sha256 sidecar before stopping the Kaggle session.")